# **LangChain Structured Output – Essential Codes Notebook**
This notebook contains **all important methods to generate structured outputs in LangChain**.

Covered topics:
- PydanticOutputParser
- JsonOutputParser
- StructuredOutputParser
- LCEL structured chains
- Function / Tool calling
- Batch structured outputs


## **1. Install Required Libraries**

In [ ]:
!pip install langchain langchain-openai pydantic

## **2. Basic LLM Setup**

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

response = llm.invoke('Explain artificial intelligence briefly')
print(response.content)

## **3. Structured Output using PydanticOutputParser**

In [ ]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name: str
    age: int
    city: str

parser = PydanticOutputParser(pydantic_object=Person)

prompt = f'''Generate a random person.
{parser.get_format_instructions()}
'''

response = llm.invoke(prompt)

result = parser.parse(response.content)

print(result)

## **4. Structured Output using JsonOutputParser**

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = '''
Generate JSON with fields: name, age, profession
'''

response = llm.invoke(prompt)

structured_data = parser.parse(response.content)

print(structured_data)

## **5. Structured Output using StructuredOutputParser**

In [ ]:
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

schemas = [
    ResponseSchema(name='name', description='person name'),
    ResponseSchema(name='age', description='person age'),
    ResponseSchema(name='profession', description='person profession')
]

parser = StructuredOutputParser.from_response_schemas(schemas)

format_instructions = parser.get_format_instructions()

prompt = f'''Generate a person.
{format_instructions}
'''

response = llm.invoke(prompt)

result = parser.parse(response.content)

print(result)

## **6. Structured Output using LCEL Chain**

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = PromptTemplate(
    template='Generate structured data about {topic} in JSON format',
    input_variables=['topic']
)

chain = prompt | llm | parser

result = chain.invoke({'topic': 'Machine Learning'})

print(result)

## **7. Structured Output with Pydantic + LCEL**

In [ ]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate

class Book(BaseModel):
    title: str
    author: str
    year: int

parser = PydanticOutputParser(pydantic_object=Book)

prompt = PromptTemplate(
    template='Generate book data.\n{format_instructions}',
    input_variables=[],
    partial_variables={'format_instructions': parser.get_format_instructions()}
)

chain = prompt | llm | parser

result = chain.invoke({})

print(result)

## **8. Structured Output using Tool / Function Calling**

In [ ]:
schema = {
    'name': 'get_user',
    'description': 'Generate user information',
    'parameters': {
        'type': 'object',
        'properties': {
            'name': {'type': 'string'},
            'age': {'type': 'integer'}
        }
    }
}

response = llm.invoke(
    'Generate user data',
    tools=[schema]
)

print(response)

## **9. Batch Structured Outputs**

In [ ]:
inputs = [
    'Generate a person JSON',
    'Generate another person JSON'
]

responses = llm.batch(inputs)

for r in responses:
    print(r.content)